# Customer Integration Validation

**Owner:** Hazim Ali  
**Assigned reviewer:** Shreyansh Pankaj  
**Run after:** `03_gold_eda.ipynb`

Validates customer uniqueness, customer-to-order reconciliation, and the repeat-customer KPI.

This notebook is an owner-specific PySpark contribution. The owner must run it personally, inspect the displayed result, understand every assertion, and commit it from their own GitHub account.


## 1. Load the validated project tables


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

customers = spark.table(f"{CATALOG}.{SCHEMA}.customers_silver")
orders_gold = spark.table(f"{CATALOG}.{SCHEMA}.orders_gold")
customer_behavior = spark.table(f"{CATALOG}.{SCHEMA}.customer_behavior_gold")
kpi = spark.table(f"{CATALOG}.{SCHEMA}.kpi_gold")

print(f"customers_silver: {customers.count():,}")
print(f"orders_gold: {orders_gold.count():,}")
print(f"customer_behavior_gold: {customer_behavior.count():,}")


## 2. Run owner-specific reconciliation and integrity checks


In [ ]:
# Customer IDs must remain unique after Silver cleaning.
assert customers.select("CustomerID").distinct().count() == customers.count()

# Every purchasing customer in Gold must exist in the cleaned customer table.
assert customer_behavior.join(
    customers.select("CustomerID"), "CustomerID", "left_anti"
).count() == 0

# Recompute each customer's completed-order count directly from orders_gold.
expected_orders = orders_gold.groupBy("CustomerID").agg(
    F.countDistinct("OrderID").alias("ExpectedOrders")
)

customer_reconciliation = customer_behavior.join(
    expected_orders, "CustomerID", "full"
).filter(
    F.coalesce(F.col("TotalOrders"), F.lit(-1))
    != F.coalesce(F.col("ExpectedOrders"), F.lit(-2))
)
assert customer_reconciliation.count() == 0

# Independently recalculate repeat-customer rate and compare it with kpi_gold.
recomputed_repeat_rate = customer_behavior.agg(
    F.round(
        F.avg(F.col("IsRepeatCustomer").cast("double")) * 100, 2
    ).alias("RecomputedRepeatCustomerRatePct")
).first()["RecomputedRepeatCustomerRatePct"]

published_repeat_rate = kpi.first()["RepeatCustomerRatePct"]
assert abs(recomputed_repeat_rate - published_repeat_rate) < 0.01


## 3. Display the observed business result and success marker


In [ ]:
customer_summary = customer_behavior.agg(
    F.countDistinct("CustomerID").alias("PurchasingCustomers"),
    F.sum(F.col("IsRepeatCustomer").cast("int")).alias("RepeatCustomers"),
    F.round(F.avg("TotalOrders"), 2).alias("AverageOrdersPerCustomer"),
    F.round(F.avg("AverageOrderValue"), 2).alias("AverageCustomerAOV"),
)
display(customer_summary)
print(f"Published repeat-customer rate: {published_repeat_rate:.2f}%")
print(f"Recomputed repeat-customer rate: {recomputed_repeat_rate:.2f}%")

print("HAZIM_CUSTOMER_VALIDATION_PASSED")


## What the owner must be able to explain

- Which tables were compared and why.
- What each assertion protects against.
- What the displayed result means for FreshRoute.
- Why the final success marker `HAZIM_CUSTOMER_VALIDATION_PASSED` only prints after every check passes.
